<a href="https://colab.research.google.com/github/Alime21/NLP_Author_Detection/blob/main/Alime_K%C4%B1l%C4%B1n%C3%A7_220709074.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: LIBRARIES AND DRIVE INSTALLATION
# ==========================================
# 1. Connect Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Main Directory path
import os
base_path = '/content/drive/MyDrive/nlp_project'

# 3. Making folders
for folder in ['data', 'vocab', 'models']:
    os.makedirs(f'{base_path}/{folder}', exist_ok=True)


In [ ]:
# ==========================================
# CELL 2:  DATA DOWNLOAD AND CLEANING
# ==========================================
import requests
import re
import pickle
import nltk
from nltk.tokenize import word_tokenize
from collections import defaultdict

# NLTK packets
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

AUTHORS = {
    "Mark Twain": [74, 76, 1837, 86, 245],
    "Jane Austen": [1342, 161, 158, 121, 105],
    "Arthur Conan Doyle": [2852, 244, 2097, 1661, 834],
    "H.G. Wells": [35, 36, 5230, 159, 1013]
}

def get_clean_text(book_id):
  """Download the book from Gutenberg and delete the license text."""
  url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
  resp = requests.get(url)
  resp.encoding = 'utf-8'
  text = resp.text

  start = re.search(r'\*\*\* START OF THE PROJECT GUTENBERG.*?\*\*\*', text)
  end = re.search(r'\*\*\* END OF THE PROJECT GUTENBERG.*?\*\*\*', text)
  if start and end:
        text = text[start.end():end.start()]
  return text

def chunk_text(text, chunk_size=500):
  """Remove punctuation from the text and divide it into 500-word blocks"""
  tokens = [w for w in word_tokenize(text.lower()) if w.isalnum()]
  chunks = []
  for i in range(0, len(tokens), chunk_size):
    chunk = tokens[i:i+chunk_size]
    if len(chunk) == chunk_size:
      chunks.append(chunk)
  return chunks

base_path = '/content/drive/MyDrive/nlp_project'
dataset = defaultdict(list)

for author, ids in AUTHORS.items():
  print(f"-> {author} books are downloading")
  for b_id in ids:
    raw_text = get_clean_text(b_id)
    dataset[author].extend(chunk_text(raw_text))

  dataset[author] = dataset[author][:200]
  print(f"   [{author}] completed! Sum {len(dataset[author])} track is ready.")

# Save all raw data to the passages.pkl file
with open(f'{base_path}/data/passages.pkl', 'wb') as f:
  pickle.dump(dataset, f)

print("\n[SUCCESS] All texts have been downloaded, tokenized, and saved to Drive as 'data/passages.pkl'!")



In [ ]:
# ==========================================
# CELL 3: DATA SPLITTING (STRATIFIED SPLIT)
# ==========================================
import pickle
import random

base_path = '/content/drive/MyDrive/nlp_project'
with open(f'{base_path}/data/passages.pkl', 'rb') as f:
  dataset = pickle.load(f)

random.seed(42)

train_corpus, train_labels = [], []
val_corpus, val_labels = [], []
test_corpus, test_labels = [], []

for author, passages in dataset.items():
  # Randomly shuffle 200 texts by each author within themselves
  random.shuffle(passages)

  # Proportions of the 200 pieces: 140 (70%), 20 (10%), 40 (20%)
  train_end = 140
  val_end = 160

  train_passages = passages[:train_end]
  val_passages = passages[train_end:val_end]
  test_passages = passages[val_end:]

  # Add Train lists
  train_corpus.extend(train_passages)
  train_labels.extend([author] * len(train_passages))

  # Add Val lists
  val_corpus.extend(val_passages)
  val_labels.extend([author] * len(val_passages))

  # Add Test lists
  test_corpus.extend(test_passages)
  test_labels.extend([author] * len(test_passages))

# To prevent the AI writers from memorizing the sets in order, shuffle all sets with their authors
def shuffle_together(corpus, labels):
    combined = list(zip(corpus, labels))
    random.shuffle(combined)
    return [c[0] for c in combined], [c[1] for c in combined]

train_corpus, train_labels = shuffle_together(train_corpus, train_labels)
val_corpus, val_labels = shuffle_together(val_corpus, val_labels)
test_corpus, test_labels = shuffle_together(test_corpus, test_labels)


with open(f'{base_path}/data/train.pkl', 'wb') as f:
    pickle.dump((train_corpus, train_labels), f)

with open(f'{base_path}/data/val.pkl', 'wb') as f:
    pickle.dump((val_corpus, val_labels), f)

with open(f'{base_path}/data/test.pkl', 'wb') as f:
    pickle.dump((test_corpus, test_labels), f)

print("\n[SUCCESS] Stratified Split completed!")
print(f"Training Set: {len(train_corpus)} items (Expected: 560)")
print(f"Validation Set: {len(val_corpus)} pieces (Expected: 80)")
print(f"Test Set: {len(test_corpus)} pieces (Expected: 160)")

